## Context

The PURE system is highly sensitive to free Mg²⁺ concentration: magnesium stabilizes ribosome structure and participates directly in peptidyl transfer, but excess Mg²⁺ inhibits aminoacyl-tRNA synthetase activity and promotes RNA aggregation. The standard PURExpress protocol specifies 9 mM Mg(OAc)₂, a value optimized for bulk solution conditions. When PURE system is used in crowded environments — such as inside lipid vesicles or in the presence of macromolecular crowding agents like PEG — the effective Mg²⁺ concentration shifts due to reduced water activity and altered diffusion constants.

This devnote describes a small-scale factorial screen intended to identify the Mg²⁺ × PEG-8000 concentration combination that maximizes eGFP fluorescence output from a 10 µL PURE system reaction at 37 °C, 2 hours. These data inform the encapsulation formulation for the Nucleus synthetic cell chassis.


## Methods

**Reaction setup.** PURExpress Δ(aa, tRNA) (NEB #E6840) reactions were set up in 10 µL volumes in 384-well plates (Corning 3544, low-volume, non-binding). Each condition was run in triplicate. Mg(OAc)₂ was added from a 100 mM stock to final concentrations of 6, 9, 12, and 15 mM. PEG-8000 (Sigma-Aldrich) was added from a 40% (w/v) stock to final concentrations of 0, 2, 4, and 6% (w/v).

**Template.** Linear dsDNA encoding eGFP under a T7 promoter (generated by PCR from pET-28a-eGFP) was added at 20 nM final concentration. DNA quality was verified by gel electrophoresis prior to use.

**Incubation and readout.** Plates were sealed with optically clear film and incubated at 37 °C in a plate reader (BioTek Synergy H1). eGFP fluorescence (Ex 485 nm / Em 528 nm) was measured every 10 minutes for 120 minutes. Final endpoint fluorescence (t = 120 min) was used for condition ranking. Background fluorescence from no-template controls was subtracted.

**Statistics.** Endpoint fluorescence values were log-transformed prior to two-way ANOVA. Tukey HSD post-hoc tests were applied for pairwise comparisons. p < 0.05 considered significant.


## Results

The optimal condition was 12 mM Mg²⁺ with 4% PEG-8000, producing a mean endpoint fluorescence of 3,420 ± 180 AU — 1.86× higher than the standard condition (9 mM Mg²⁺, 0% PEG). The 12 mM / 4% PEG condition was statistically distinguishable from all other conditions (Tukey HSD, p < 0.001).

Key observations:
- PEG-8000 alone (at 9 mM Mg²⁺) increased yield by ~30% at 2–4%, consistent with macromolecular crowding enhancing ribosome assembly kinetics.
- At 6% PEG, yield dropped sharply, likely due to crowding-induced inhibition of tRNA diffusion to the ribosome A-site.
- Higher Mg²⁺ (15 mM) reduced yield at all PEG concentrations, suggesting aminoacyl-tRNA synthetase inhibition dominates at elevated free Mg²⁺.
- Expression kinetics showed that the optimal condition reached half-maximal fluorescence ~15 min earlier than the standard condition, indicating faster ribosome assembly or elongation.


## Specification

| Parameter | Value |
|---|---|
| PURE system kit | PURExpress Δ(aa, tRNA), NEB #E6840 |
| Reaction volume | 10 µL |
| Plate format | 384-well, Corning 3544 |
| Mg(OAc)₂ concentrations tested | 6, 9, 12, 15 mM |
| PEG-8000 concentrations tested | 0, 2, 4, 6% (w/v) |
| DNA template | eGFP, linear, T7 promoter, 20 nM |
| Incubation temperature | 37 °C |
| Incubation duration | 120 min |
| Fluorescence excitation | 485 nm |
| Fluorescence emission | 528 nm |
| Measurement interval | 10 min |
| Replicates per condition | 3 |
| Total conditions | 16 (4×4 factorial) |
| Optimal Mg²⁺ | 12 mM |
| Optimal PEG-8000 | 4% (w/v) |
| Fold improvement over standard | 1.86× |


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import os

os.makedirs('figures', exist_ok=True)

rng = np.random.default_rng(7)
time = np.linspace(0, 120, 13)  # 0 to 120 min, 10 min intervals

# Kinetics for each Mg2+ condition at optimal 4% PEG
conditions = {
    '6 mM Mg²⁺': {'plateau': 1600, 'k': 0.045, 'lag': 12, 'color': '#94A3B8'},
    '9 mM Mg²⁺ (standard)': {'plateau': 1840, 'k': 0.055, 'lag': 10, 'color': '#60A5FA'},
    '12 mM Mg²⁺ (optimal)': {'plateau': 3420, 'k': 0.075, 'lag': 6, 'color': '#1D4ED8'},
    '15 mM Mg²⁺': {'plateau': 1200, 'k': 0.040, 'lag': 14, 'color': '#F87171'},
}

def kinetics(t, plateau, k, lag):
    t_adj = np.maximum(t - lag, 0)
    return plateau * (1 - np.exp(-k * t_adj))

mpl.rcParams.update({'font.size': 11, 'axes.spines.top': False, 'axes.spines.right': False})

fig, ax = plt.subplots(figsize=(7.5, 4.5))
for label, params in conditions.items():
    trace = kinetics(time, params['plateau'], params['k'], params['lag'])
    noise = rng.normal(0, params['plateau'] * 0.02, len(time))
    lw = 2.8 if 'optimal' in label else 1.8
    ls = '-' if 'optimal' in label else '--'
    ax.plot(time, trace + noise, color=params['color'], linewidth=lw, linestyle=ls, label=label)

ax.set_xlabel('Time (min)', labelpad=8)
ax.set_ylabel('eGFP Fluorescence Intensity (AU)', labelpad=8)
ax.set_title('PURE System eGFP Expression at 4% PEG-8000\nVarying Mg(OAc)₂ Concentration', fontsize=11, pad=10)
ax.set_xlim(0, 120)
ax.set_ylim(0, 4000)
ax.legend(frameon=False, fontsize=9, loc='upper left')
plt.tight_layout()
plt.savefig('figures/key-result.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved figures/key-result.png')
